In [1]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)

In [2]:
import pymupdf as pp

In [3]:
doc = pp.open("data/credit_policy_extract.pdf")
doc

Document('data/credit_policy_extract.pdf')

In [4]:
page = doc[0]

In [5]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter(show_progress=True)

c:\Users\apaks\projects\RiskLens2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
file_path = "data/credit_policy_extract_2.pdf"
docs = converter.convert(file_path)

[1] Converting credit_policy_extract_2.pdf
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1195.30it/s]
  pages 34/34
Finished credit_policy_extract_2.pdf: success


In [9]:
doc = docs.document

In [11]:
for item, level in doc.iterate_items():
    print("item: ", item.label)
    print("level: ", item.text if hasattr(item, "text") else "table")



item:  section_header
level:  Northbridge Bank plc - SME Credit Policy
item:  text
level:  FICTIONAL DOCUMENT. Northbridge Bank plc does not exist. This policy, and every threshold, limit, scorecard weight, PD, LGD, authority level and worked example in it, is invented for educational and portfolio-project purposes. It is not, and must not be presented as, the policy of any real institution.
item:  section_header
level:  6. Eligibility
item:  text
level:  An application that fails a mandatory eligibility criterion must be declined or approved as a Policy Exception; it may not proceed to credit assessment as if eligible. Eligibility is tested by the decision engine for engine-routed applications and by the RM (checked by Credit) for all others.
item:  section_header
level:  6.1 Mandatory eligibility criteria
item:  table
level:  table
item:  table
level:  table
item:  section_header
level:  6.2 Sector appetite classification
item:  text
level:  Northbridge classifies sectors by SIC 2007

In [40]:
from docling.chunking import HierarchicalChunker

chunker = HierarchicalChunker()

chunks = chunker.chunk(doc)

In [41]:
import pickle

with open("data/artifacts/docling_chunks.pkl", "wb") as f:
    pickle.dump(list(chunks), f)

In [42]:
with open("data/artifacts/docling_chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

In [44]:
chunks

[DocChunk(text='FICTIONAL DOCUMENT. Northbridge Bank plc does not exist. This policy, and every threshold, limit, scorecard weight, PD, LGD, authority level and worked example in it, is invented for educational and portfolio-project purposes. It is not, and must not be presented as, the policy of any real institution.', meta=DocMeta(schema_name='docling_core.transforms.chunker.DocMeta', version='1.0.0', doc_items=[TextItem(self_ref='#/texts/1', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=50.72450572647884, t=726.604480349091, r=536.6894232050387, b=694.4103571772503, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 303))], source=[], comments=[], orig='FICTIONAL DOCUMENT. Northbridge Bank plc does not exist. This policy, and every threshold, limit, scorecard weight, PD, LGD, authority level and worked example in it, is invented fo

In [26]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: \n", chunk.text)
    print(f"Chunk-Metadata: ", chunk.meta)
    print()

Chunk 1: 
 FICTIONAL DOCUMENT. Northbridge Bank plc does not exist. This policy, and every threshold, limit, scorecard weight, PD, LGD, authority level and worked example in it, is invented for educational and portfolio-project purposes. It is not, and must not be presented as, the policy of any real institution.
Chunk-Metadata:  schema_name='docling_core.transforms.chunker.DocMeta' version='1.0.0' doc_items=[TextItem(self_ref='#/texts/1', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=50.72450572647884, t=726.604480349091, r=536.6894232050387, b=694.4103571772503, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 303))], source=[], comments=[], orig='FICTIONAL DOCUMENT. Northbridge Bank plc does not exist. This policy, and every threshold, limit, scorecard weight, PD, LGD, authority level and worked example in it, is invented for edu

In [45]:
import uuid

final_chunks = []

def extract_chunk_type(chunk):

    labels = {
        item.label.value
        for item in chunk.meta.doc_items
    }

    if "table" in labels:
        return "table"

    if "text" in labels:
        return "text"

    return "other"


for i, chunk in enumerate(chunks):
    content = chunk.text
    chunk_metadata = chunk.meta

    metadata = {
        "chunk_id": str(uuid.uuid4()),
        "document_name": chunk_metadata.origin.filename,
        "section": chunk_metadata.headings or [],
        "page": chunk_metadata.doc_items[0].prov[0].page_no,
        "content_type": extract_chunk_type(chunk)
    }

    final_chunks.append(
        {
            "content": content,
            "metadata": metadata
        }
    )



In [46]:
final_chunks

[{'content': 'FICTIONAL DOCUMENT. Northbridge Bank plc does not exist. This policy, and every threshold, limit, scorecard weight, PD, LGD, authority level and worked example in it, is invented for educational and portfolio-project purposes. It is not, and must not be presented as, the policy of any real institution.',
  'metadata': {'chunk_id': '228fdd2c-b0aa-4e4d-bf37-a9f41bd7d26e',
   'document_name': 'credit_policy_extract_2.pdf',
   'section': ['Northbridge Bank plc - SME Credit Policy'],
   'page': 1,
   'content_type': 'text'}},
 {'content': 'An application that fails a mandatory eligibility criterion must be declined or approved as a Policy Exception; it may not proceed to credit assessment as if eligible. Eligibility is tested by the decision engine for engine-routed applications and by the RM (checked by Credit) for all others.',
  'metadata': {'chunk_id': '6b098ef0-2c6f-4978-aea3-8478761ba080',
   'document_name': 'credit_policy_extract_2.pdf',
   'section': ['6. Eligibility'